# Flow matching for simple unsupervised distributions

In [ ]:
%load_ext autoreload
%autoreload 2

import data  # Auxiliary module for generating toy datasets
import models  # Auxiliary module for neural network architecture and training
import plotting  # Auxiliary module for visualizations

## Prepare toy data

Generate target distribution data and original noisy data (a standard gaussian)

In [ ]:
import numpy as np

n = 1000
toy_dataset = "two_gaussians"

source_data = np.random.randn(n, 2)

target_data = data.generate_toy_data(toy_dataset, n)

plotting.plot_distributions(source_data, target_data)

## Basic flow matching

Let's sample independent couplings between source and target data points

In [ ]:
couplings = models.sample_independent_couplings(source_data, target_data, num_couplings=n)

Visualize the generated couplings

In [ ]:
plotting.plot_distributions(source_data, target_data, couplings=couplings)

Trains a simple MLP as the velocity field estimation network.

In [ ]:
network = models.train_flow_model(couplings, verbose=True)

Visualize the velocity field we have learned

In [ ]:
plotting.plot_velocity_field(network, source_data, target_data)

Now that we have the velocity field, we can use a integration method such as Euler's method to run across the flow.

In [ ]:
trajectories = models.compute_trajectories(network, source_data)
plotting.animate_trajectories(trajectories, target_data=target_data)

## Rectified flows

By using the generated trajectories to learn a new flow, we can improve the "straightness" of the trajectories, allowing for faster generation.

In [ ]:
synthetic_data = np.array([traj[-1][1] for traj in trajectories])
couplings = [(src, tgt) for src, tgt in zip(source_data, synthetic_data)]
plotting.plot_distributions(source_data, synthetic_data, couplings=couplings)

In [ ]:
rectified_network = models.train_flow_model(couplings, verbose=True)

Let's check now the rectified velocity field

In [ ]:
plotting.plot_velocity_field(rectified_network, source_data, target_data)

The trajectories should be straigther in the new field

In [ ]:
rectified_trajectories = models.compute_trajectories(rectified_network, source_data)
plotting.animate_trajectories(rectified_trajectories, target_data=target_data)

In [ ]:
plotting.plot_generated_data_comparison(target_data, trajectories)

## Reverse trajectories

Since the flow is reversible, we can compute reversed trajectories to see how data from the target distribution can be transformed into data from the original distribution.

In [ ]:
reversed_trajectories = models.compute_trajectories(rectified_network, target_data, reverse=True)
plotting.animate_trajectories(reversed_trajectories, target_data=source_data)

This is particularly useful to compute the degree of anomaly of new data points, as usually the original distribution follows a closed equation for its probability distribution. We can test this with a mesh of points, running their reversed trajectories and then assigning to each starting point a "normality" score following the PDF of the gaussian in the trajectory endpoints.

In [ ]:
mesh_trajectories = models.compute_trajectories(rectified_network, plotting.mesh_from_data(target_data), reverse=True)
plotting.animate_trajectories(mesh_trajectories)

In [ ]:
plotting.plot_density_map(mesh_trajectories, target_data)